In [22]:
import os
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
import numpy as np
import editdistance

In [23]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [24]:
base_dir = os.getcwd()

ground_truth_path = os.path.join(base_dir, '/kaggle/input/balinese/data/balinese_transliteration_train.txt') 

images_dir = os.path.join(base_dir, '/kaggle/input/balinese/data/balinese_word_train')

In [25]:
filenames = []
labels = []

# Read the ground truth text file
with open(ground_truth_path, 'r', encoding='utf-8') as file:
    for line in file:
        line = line.strip()
        if line:  # Ensure the line is not empty
            parts = line.split(';')
            if len(parts) == 2:
                filename, label = parts
                filenames.append(filename)
                labels.append(label)
            else:
                print(f"Skipping malformed line: {line}")

# Create a DataFrame from the data
data = pd.DataFrame({
    'filename': filenames,
    'label': labels
})

# Display the DataFrame
data.head()


,filename,label
0,train1.png,kagastu
1,train2.png,","
2,train3.png,gelak
3,train4.png,ancak
4,train5.png,","


In [26]:
# Build a character vocabulary from the labels
all_text = ''.join(data['label'])  # Concatenate all labels into one string
chars = sorted(list(set(all_text)))  # Extract unique characters and sort them

# Create character to index mapping, starting from index 1
char_to_idx = {char: idx + 1 for idx, char in enumerate(chars)}  # Start indexing from 1

# Add special tokens
char_to_idx['<PAD>'] = 0  # Add padding token at index 0
char_to_idx['<UNK>'] = len(char_to_idx)  # Add unknown token
char_to_idx['<SOS>'] = len(char_to_idx)  # Add start-of-sequence token
char_to_idx['<EOS>'] = len(char_to_idx)  # Add end-of-sequence token

# Create index to character mapping
idx_to_char = {idx: char for char, idx in char_to_idx.items()}

# Update vocabulary size
vocab_size = len(char_to_idx)

print(f"Vocabulary size: {vocab_size}")


Vocabulary size: 58


In [27]:
# print("Character to Index Mapping:")
# for char, idx in char_to_idx.items():
#     print(f"'{char}': {idx}")


In [28]:
empty_labels = data[data['label'].str.strip() == '']
print(f"Number of empty labels in training data: {len(empty_labels)}")

Number of empty labels in training data: 0


In [29]:
# add padding to all labels to ensure all labels have same length, use index of character to represent the label
# Encode labels as sequences of character indices
max_label_length = max(len(label) for label in data['label']) + 2  # +2 for <SOS> and <EOS>

# def encode_label(label, char_to_idx, max_length):
#     encoded = [char_to_idx[char] for char in label]
#     # Pad the encoded label
#     padded = encoded + [char_to_idx['<PAD>']] * (max_length - len(encoded))
#     return padded

def encode_label(label, char_to_idx, max_length):
    encoded = (
        [char_to_idx['<SOS>']] +
        [char_to_idx.get(char, char_to_idx['<UNK>']) for char in label] +
        [char_to_idx['<EOS>']]
    )
    padded = encoded + [char_to_idx['<PAD>']] * (max_length - len(encoded))
    return padded


data['encoded_label'] = data['label'].apply(lambda x: encode_label(x, char_to_idx, max_label_length))



In [30]:
# Split the dataset into training and validation sets
train_data, val_data = train_test_split(data, test_size=0.1, random_state=42)

In [31]:
# Define a custom dataset class
# Image data
class BalineseDataset(Dataset):
    def __init__(self, data, images_dir, transform=None):
        self.data = data.reset_index(drop=True)
        self.images_dir = images_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        """given index, return corresponding image and label"""
        img_name = self.data.loc[idx, 'filename']
        label = self.data.loc[idx, 'encoded_label']
        img_path = os.path.join(self.images_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(label, dtype=torch.long)
        return image, label


In [32]:
# Define image transformations
transform = transforms.Compose([
    transforms.Resize((64, 256)),  # Resize images to a fixed size
    transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5),  # Mean for R, G, B channels
                            (0.5, 0.5, 0.5))  # Std for R, G, B channels  # Normalize images
])

# Create dataset instances
train_dataset = BalineseDataset(train_data, images_dir, transform=transform)
val_dataset = BalineseDataset(val_data, images_dir, transform=transform)


In [33]:
# Create data loaders
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)


In [34]:
# Define the Encoder
class EncoderCNN(nn.Module):
    def __init__(self, encoded_image_size=14):
        super(EncoderCNN, self).__init__()
        self.enc_image_size = encoded_image_size

        # Pretrained ResNet model
        resnet = models.resnet18(pretrained=True)
        modules = list(resnet.children())[:-2]  # Remove the last two layers (avgpool and fc)
        self.resnet = nn.Sequential(*modules)

        # Resize the encoded images to a fixed size
        self.adaptive_pool = nn.AdaptiveAvgPool2d((encoded_image_size, encoded_image_size))

    def forward(self, images):
        features = self.resnet(images)  # [batch_size, 512, h/32, w/32]
        features = self.adaptive_pool(features)  # [batch_size, 512, encoded_image_size, encoded_image_size]
        # Flatten the features to (batch_size, num_pixels, encoder_dim)
        features = features.permute(0, 2, 3, 1)  # [batch_size, encoded_image_size, encoded_image_size, 512]
        features = features.view(features.size(0), -1, features.size(-1))  # [batch_size, num_pixels, encoder_dim]
        return features


In [35]:
# Define the Attention Mechanism
class Attention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super(Attention, self).__init__()
        self.encoder_att = nn.Linear(encoder_dim, attention_dim)  # Linear layer to transform encoder output
        self.decoder_att = nn.Linear(decoder_dim, attention_dim)  # Linear layer to transform decoder hidden state
        self.full_att = nn.Linear(attention_dim, 1)  # Linear layer to compute attention scores
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)  # Softmax to compute weights

    def forward(self, encoder_out, decoder_hidden):
        # encoder_out: [batch_size, num_pixels, encoder_dim]
        # decoder_hidden: [batch_size, decoder_dim]
        att1 = self.encoder_att(encoder_out)  # [batch_size, num_pixels, attention_dim]
        att2 = self.decoder_att(decoder_hidden)  # [batch_size, attention_dim]
        att = self.full_att(self.relu(att1 + att2.unsqueeze(1))).squeeze(2)  # [batch_size, num_pixels]
        alpha = self.softmax(att)  # [batch_size, num_pixels]
        # convex vector
        attention_weighted_encoding = (encoder_out * alpha.unsqueeze(2)).sum(dim=1)  # [batch_size, encoder_dim]
        return attention_weighted_encoding, alpha


In [36]:
# Define the Decoder
class DecoderRNN(nn.Module):
    def __init__(self, attention_dim, embed_dim, decoder_dim, vocab_size, encoder_dim=512):
        super(DecoderRNN, self).__init__()
        self.attention = Attention(encoder_dim, decoder_dim, attention_dim)  # Attention network

        self.embedding = nn.Embedding(vocab_size, embed_dim)  # Embedding layer
        self.dropout = nn.Dropout(p=0.5)
        self.decode_step = nn.LSTMCell(embed_dim + encoder_dim, decoder_dim, bias=True)  # LSTMCell
        self.init_h = nn.Linear(encoder_dim, decoder_dim)  # Linear layer to initialize hidden state
        self.init_c = nn.Linear(encoder_dim, decoder_dim)  # Linear layer to initialize cell state
        self.f_beta = nn.Linear(decoder_dim, encoder_dim)  # Linear layer to create a gating scalar
        self.sigmoid = nn.Sigmoid()
        self.fc = nn.Linear(decoder_dim, vocab_size)  # Linear layer to find scores over vocabulary
        self.init_weights()  # Initialize some layers with the uniform distribution

    def init_weights(self):
        """Initialize weights."""
        self.embedding.weight.data.uniform_(-0.1, 0.1)
        self.fc.bias.data.fill_(0)
        self.fc.weight.data.uniform_(-0.1, 0.1)

    def init_hidden_state(self, encoder_out):
        """Create the initial hidden and cell states for the decoder's LSTM based on the encoded images."""
        mean_encoder_out = encoder_out.mean(dim=1)
        h = self.init_h(mean_encoder_out)  # [batch_size, decoder_dim]
        c = self.init_c(mean_encoder_out)
        return h, c

    def forward(self, encoder_out, encoded_captions, caption_lengths):
        """
        Forward propagation.
        :param encoder_out: encoded images, [batch_size, num_pixels, encoder_dim]
        :param encoded_captions: encoded captions, [batch_size, max_caption_length]
        :param caption_lengths: caption lengths, [batch_size, 1]
        """
        batch_size = encoder_out.size(0)
        encoder_dim = encoder_out.size(-1)
        vocab_size = self.fc.out_features

        num_pixels = encoder_out.size(1)

        # Sort input data by decreasing lengths; why? pack_padded_sequence requires it
        caption_lengths, sort_ind = caption_lengths.squeeze(1).sort(dim=0, descending=True)
        encoder_out = encoder_out[sort_ind]
        encoded_captions = encoded_captions[sort_ind]

        # Embedding
        embeddings = self.embedding(encoded_captions)  # [batch_size, max_caption_length, embed_dim]

        # Initialize LSTM state
        h, c = self.init_hidden_state(encoder_out)  # [batch_size, decoder_dim]

        # We won't decode at the <PAD> positions, so create a packed sequence
        decode_lengths = (caption_lengths - 1).tolist()

        # Create tensors to hold word predictions and alphas
        max_decode_length = max(decode_lengths)
        predictions = torch.zeros(batch_size, max_decode_length, vocab_size).to(encoder_out.device)
        alphas = torch.zeros(batch_size, max_decode_length, num_pixels).to(encoder_out.device)

        # At each time-step, decode by attention-weighting the encoder's output based on the decoder's previous hidden state
        for t in range(max_decode_length):
            batch_size_t = sum([l > t for l in decode_lengths])
            attention_weighted_encoding, alpha = self.attention(
                encoder_out[:batch_size_t], h[:batch_size_t]
            )
            gate = self.sigmoid(self.f_beta(h[:batch_size_t]))  # gating scalar
            attention_weighted_encoding = gate * attention_weighted_encoding

            h, c = self.decode_step(
                torch.cat([embeddings[:batch_size_t, t, :], attention_weighted_encoding], dim=1),
                (h[:batch_size_t], c[:batch_size_t])
            )  # LSTMCell

            preds = self.fc(self.dropout(h))  # [batch_size_t, vocab_size]
            predictions[:batch_size_t, t, :] = preds
            alphas[:batch_size_t, t, :] = alpha

        return predictions, encoded_captions, decode_lengths, alphas, sort_ind


In [37]:
# Instantiate the encoder and decoder
encoder = EncoderCNN()
decoder = DecoderRNN(
    attention_dim=256,
    embed_dim=256,
    decoder_dim=512,
    vocab_size=vocab_size,
    encoder_dim=512
)

# Move models to device
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = encoder.to(device)
decoder = decoder.to(device)
print(device)


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


cuda


In [38]:
# Define loss function and optimizers
criterion = nn.CrossEntropyLoss(ignore_index=char_to_idx['<PAD>'])
encoder_optimizer = optim.Adam(encoder.parameters(), lr=1e-4)
decoder_optimizer = optim.Adam(decoder.parameters(), lr=4e-4)

In [39]:
class ImageCaptioningTrainer:
    def __init__(self, encoder, decoder, 
                 criterion, encoder_optimizer, decoder_optimizer, 
                 train_loader, val_loader, device, 
                 char_to_idx, idx_to_char, max_label_length):
        self.encoder = encoder
        self.decoder = decoder
        self.criterion = criterion
        self.encoder_optimizer = encoder_optimizer
        self.decoder_optimizer = decoder_optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.char_to_idx = char_to_idx
        self.idx_to_char = idx_to_char
        self.max_label_length = max_label_length

    # def fit(self, num_epochs):
    #     for epoch in range(num_epochs):
    #         print(f'Epoch {epoch + 1}/{num_epochs}')
    #         train_loss, train_cer = self.train_one_epoch()
    #         val_loss, val_cer = self.validate_one_epoch()
    #         print(f'Epoch [{epoch +1}/{num_epochs}] - '
    #             f'Train Loss: {train_loss:.4f}, Train CER: {train_cer:.4f} - '
    #             f'Val Loss: {val_loss:.4f}, Val CER: {val_cer:.4f}\n')
    def fit(self, num_epochs):
        for epoch in range(num_epochs):
            print(f'Epoch {epoch + 1}/{num_epochs}')
            train_loss, train_cer = self.train_one_epoch()
            val_loss, val_cer = self.validate_one_epoch()
            print(f'Epoch [{epoch +1}/{num_epochs}] - '
                f'Train Loss: {train_loss:.4f}, Train CER: {train_cer:.4f} - '
                f'Val Loss: {val_loss:.4f}, Val CER: {val_cer:.4f}\n')

    
    def train_one_epoch(self):
        self.encoder.train()
        self.decoder.train()
        running_loss = 0.0
        total_cer = 0.0
        total_samples = 0

        for batch_idx, (images, labels) in enumerate(self.train_loader):
            images = images.to(device)
            labels = labels.to(device)

            # Zero the gradients
            self.encoder_optimizer.zero_grad()
            self.decoder_optimizer.zero_grad()

            # Forward pass
            encoder_out = self.encoder(images)
            caption_lengths = torch.tensor(
                [self.max_label_length] * labels.size(0)
            ).unsqueeze(1).to(device)
            outputs, encoded_captions, decode_lengths, alphas, sort_ind = self.decoder(
                encoder_out, labels, caption_lengths
            )

            # Reshape targets
            targets = encoded_captions[:, 1:]

            # Flatten outputs and targets for loss computation
            outputs_flat = outputs.view(-1, self.decoder.fc.out_features)
            targets_flat = targets.contiguous().view(-1)

            # Compute loss
            loss = self.criterion(outputs_flat, targets_flat)
            loss.backward()

            # Update weights
            self.decoder_optimizer.step()
            self.encoder_optimizer.step()

            running_loss += loss.item()

            # Compute CER for the batch
            batch_size = labels.size(0)
            _, preds_flat = torch.max(outputs_flat, dim=1)
            preds_seq = preds_flat.view(batch_size, -1)

            for i in range(batch_size):
                pred_indices = preds_seq[i].detach().cpu().numpy()
                target_indices = targets[i].detach().cpu().numpy()

                # Remove padding tokens
                mask = target_indices != self.char_to_idx['<PAD>']
                pred_indices = pred_indices[mask]
                target_indices = target_indices[mask]

                # Decode indices to characters
                pred_chars = [self.idx_to_char.get(idx, '') for idx in pred_indices]
                target_chars = [self.idx_to_char.get(idx, '') for idx in target_indices]

                # Join characters to form strings
                pred_str = ''.join(pred_chars)
                target_str = ''.join(target_chars)

                # Calculate CER for this sample
                if len(target_str) > 0:
                    cer = editdistance.eval(pred_str, target_str) / len(target_str)
                    total_cer += cer
                else:
                    total_cer += 0.0  # Avoid division by zero

            total_samples += batch_size

            if (batch_idx + 1) % 50 == 0 or (batch_idx + 1) == len(self.train_loader):
                print(f'Batch {batch_idx + 1}/{len(self.train_loader)} - Loss: {loss.item():.4f}')

        avg_loss = running_loss / len(self.train_loader)
        avg_cer = total_cer / total_samples if total_samples > 0 else 0.0
        return avg_loss, avg_cer

    def validate_one_epoch(self):
        self.encoder.eval()
        self.decoder.eval()
        running_loss = 0.0
        total_cer = 0.0
        total_samples = 0

        with torch.no_grad():
            for batch_idx, (images, labels) in enumerate(self.val_loader):
                images = images.to(device)
                labels = labels.to(device)

                # Forward pass
                encoder_out = self.encoder(images)
                caption_lengths = torch.tensor(
                    [self.max_label_length] * labels.size(0)
                ).unsqueeze(1).to(device)
                outputs, encoded_captions, decode_lengths, alphas, sort_ind = self.decoder(
                    encoder_out, labels, caption_lengths
                )

                # Reshape targets
                targets = encoded_captions[:, 1:]

                # Flatten outputs and targets for loss computation
                outputs_flat = outputs.view(-1, self.decoder.fc.out_features)
                targets_flat = targets.contiguous().view(-1)

                # Compute loss
                loss = self.criterion(outputs_flat, targets_flat)
                running_loss += loss.item()

                # Compute CER for the batch
                batch_size = labels.size(0)
                _, preds_flat = torch.max(outputs_flat, dim=1)
                preds_seq = preds_flat.view(batch_size, -1)

                for i in range(batch_size):
                    pred_indices = preds_seq[i].detach().cpu().numpy()
                    target_indices = targets[i].detach().cpu().numpy()

                    # Remove padding tokens
                    mask = target_indices != self.char_to_idx['<PAD>']
                    pred_indices = pred_indices[mask]
                    target_indices = target_indices[mask]

                    # Decode indices to characters
                    pred_chars = [self.idx_to_char.get(idx, '') for idx in pred_indices]
                    target_chars = [self.idx_to_char.get(idx, '') for idx in target_indices]

                    # Join characters to form strings
                    pred_str = ''.join(pred_chars)
                    target_str = ''.join(target_chars)

                    # Calculate CER for this sample
                    if len(target_str) > 0:
                        cer = editdistance.eval(pred_str, target_str) / len(target_str)
                        total_cer += cer
                    else:
                        total_cer += 0.0  # Avoid division by zero

                    # Print a few samples
                    if batch_idx == 0 and i < 3: 
                        print(f"Sample {i + 1}:")
                        print(f"Predicted: {pred_str}")
                        print(f"Target   : {target_str}\n")

                total_samples += batch_size

        avg_loss = running_loss / len(self.val_loader)
        avg_cer = total_cer / total_samples if total_samples > 0 else 0.0
        return avg_loss, avg_cer


In [40]:
trainer = ImageCaptioningTrainer(
    encoder=encoder,
    decoder=decoder,
    criterion=criterion,
    encoder_optimizer=encoder_optimizer,
    decoder_optimizer=decoder_optimizer,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    char_to_idx=char_to_idx,
    idx_to_char=idx_to_char,
    max_label_length=max_label_length
)

# trainer = trainer.to(device)
num_epochs = 40
trainer.fit(num_epochs)

Epoch 1/40
Batch 50/423 - Loss: 2.4003
Batch 100/423 - Loss: 2.1890
Batch 150/423 - Loss: 1.7986
Batch 200/423 - Loss: 1.7608
Batch 250/423 - Loss: 1.5511
Batch 300/423 - Loss: 1.5309
Batch 350/423 - Loss: 1.3797
Batch 400/423 - Loss: 1.4171
Batch 423/423 - Loss: 1.3061
Sample 1:
Predicted: knggeng
Target   : anggon<EOS>

Sample 2:
Predicted: kunununangng<EOS>
Target   : kaputusaning<EOS>

Sample 3:
Predicted: ,<EOS>
Target   : ,<EOS>

Epoch [1/40] - Train Loss: 1.7991, Train CER: 0.4864 - Val Loss: 1.2282, Val CER: 0.2873

Epoch 2/40
Batch 50/423 - Loss: 1.1884
Batch 100/423 - Loss: 1.0789
Batch 150/423 - Loss: 1.0786
Batch 200/423 - Loss: 1.1644
Batch 250/423 - Loss: 1.0190
Batch 300/423 - Loss: 1.0064
Batch 350/423 - Loss: 0.8870
Batch 400/423 - Loss: 0.7520
Batch 423/423 - Loss: 0.7231
Sample 1:
Predicted: anggen<EOS>
Target   : anggon<EOS>

Sample 2:
Predicted: nutunaningng<EOS>
Target   : kaputusaning<EOS>

Sample 3:
Predicted: ,<EOS>
Target   : ,<EOS>

Epoch [2/40] - Train Loss:

In [41]:
class ImageCaptioningTrainer:
    def __init__(self, encoder, decoder, 
                 criterion, encoder_optimizer, decoder_optimizer, 
                 train_loader, val_loader, device, 
                 char_to_idx, idx_to_char, max_label_length):
        self.encoder = encoder
        self.decoder = decoder
        self.criterion = criterion
        self.encoder_optimizer = encoder_optimizer
        self.decoder_optimizer = decoder_optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.char_to_idx = char_to_idx
        self.idx_to_char = idx_to_char
        self.max_label_length = max_label_length

    # def fit(self, num_epochs):
    #     for epoch in range(num_epochs):
    #         print(f'Epoch {epoch + 1}/{num_epochs}')
    #         train_loss, train_cer = self.train_one_epoch()
    #         val_loss, val_cer = self.validate_one_epoch()
    #         print(f'Epoch [{epoch +1}/{num_epochs}] - '
    #             f'Train Loss: {train_loss:.4f}, Train CER: {train_cer:.4f} - '
    #             f'Val Loss: {val_loss:.4f}, Val CER: {val_cer:.4f}\n')
    def fit(self, num_epochs):
        for epoch in range(num_epochs):
            print(f'Epoch {epoch + 1}/{num_epochs}')
            train_loss, train_cer = self.train_one_epoch()
            val_loss, val_cer = self.validate_one_epoch()
            print(f'Epoch [{epoch +1}/{num_epochs}] - '
                f'Train Loss: {train_loss:.4f}, Train CER: {train_cer:.4f} - '
                f'Val Loss: {val_loss:.4f}, Val CER: {val_cer:.4f}\n')

    
    def train_one_epoch(self):
        self.encoder.train()
        self.decoder.train()
        running_loss = 0.0
        total_cer = 0.0
        total_samples = 0

        for batch_idx, (images, labels) in enumerate(self.train_loader):
            images = images.to(device)
            labels = labels.to(device)

            # Zero the gradients
            self.encoder_optimizer.zero_grad()
            self.decoder_optimizer.zero_grad()

            # Forward pass
            encoder_out = self.encoder(images)
            caption_lengths = torch.tensor(
                [self.max_label_length] * labels.size(0)
            ).unsqueeze(1).to(device)
            outputs, encoded_captions, decode_lengths, alphas, sort_ind = self.decoder(
                encoder_out, labels, caption_lengths
            )

            # Reshape targets
            targets = encoded_captions[:, 1:]

            # Flatten outputs and targets for loss computation
            outputs_flat = outputs.view(-1, self.decoder.fc.out_features)
            targets_flat = targets.contiguous().view(-1)

            # Compute loss
            loss = self.criterion(outputs_flat, targets_flat)
            loss.backward()

            # Update weights
            self.decoder_optimizer.step()
            self.encoder_optimizer.step()

            running_loss += loss.item()

            # Compute CER for the batch
            batch_size = labels.size(0)
            _, preds_flat = torch.max(outputs_flat, dim=1)
            preds_seq = preds_flat.view(batch_size, -1)

            for i in range(batch_size):
                pred_indices = preds_seq[i].detach().cpu().numpy()
                target_indices = targets[i].detach().cpu().numpy()

                # Remove padding tokens
                mask = target_indices != self.char_to_idx['<PAD>']
                pred_indices = pred_indices[mask]
                target_indices = target_indices[mask]

                # Decode indices to characters
                pred_chars = [self.idx_to_char.get(idx, '') for idx in pred_indices]
                target_chars = [self.idx_to_char.get(idx, '') for idx in target_indices]

                # Join characters to form strings
                pred_str = ''.join(pred_chars)
                target_str = ''.join(target_chars)

                # Calculate CER for this sample
                if len(target_str) > 0:
                    cer = editdistance.eval(pred_str, target_str) / len(target_str)
                    total_cer += cer
                else:
                    total_cer += 0.0  # Avoid division by zero

            total_samples += batch_size

            if (batch_idx + 1) % 50 == 0 or (batch_idx + 1) == len(self.train_loader):
                print(f'Batch {batch_idx + 1}/{len(self.train_loader)} - Loss: {loss.item():.4f}')

        avg_loss = running_loss / len(self.train_loader)
        avg_cer = total_cer / total_samples if total_samples > 0 else 0.0
        return avg_loss, avg_cer

    def validate_one_epoch(self):
        self.encoder.eval()
        self.decoder.eval()
        running_loss = 0.0
        total_cer = 0.0
        total_samples = 0

        with torch.no_grad():
            for batch_idx, (images, labels) in enumerate(self.val_loader):
                images = images.to(device)
                labels = labels.to(device)

                # Forward pass
                encoder_out = self.encoder(images)
                caption_lengths = torch.tensor(
                    [self.max_label_length] * labels.size(0)
                ).unsqueeze(1).to(device)
                outputs, encoded_captions, decode_lengths, alphas, sort_ind = self.decoder(
                    encoder_out, labels, caption_lengths
                )

                # Reshape targets
                targets = encoded_captions[:, 1:]

                # Flatten outputs and targets for loss computation
                outputs_flat = outputs.view(-1, self.decoder.fc.out_features)
                targets_flat = targets.contiguous().view(-1)

                # Compute loss
                loss = self.criterion(outputs_flat, targets_flat)
                running_loss += loss.item()

                # Compute CER for the batch
                batch_size = labels.size(0)
                _, preds_flat = torch.max(outputs_flat, dim=1)
                preds_seq = preds_flat.view(batch_size, -1)

                for i in range(batch_size):
                    pred_indices = preds_seq[i].detach().cpu().numpy()
                    target_indices = targets[i].detach().cpu().numpy()

                    # Remove padding tokens
                    mask = target_indices != self.char_to_idx['<PAD>']
                    pred_indices = pred_indices[mask]
                    target_indices = target_indices[mask]

                    # Decode indices to characters
                    pred_chars = [self.idx_to_char.get(idx, '') for idx in pred_indices]
                    target_chars = [self.idx_to_char.get(idx, '') for idx in target_indices]

                    # Join characters to form strings
                    pred_str = ''.join(pred_chars)
                    target_str = ''.join(target_chars)

                    # Calculate CER for this sample
                    if len(target_str) > 0:
                        cer = editdistance.eval(pred_str, target_str) / len(target_str)
                        total_cer += cer
                    else:
                        total_cer += 0.0  # Avoid division by zero

                    # Print a few samples
                    if batch_idx == 0 and i < 3: 
                        print(f"Sample {i + 1}:")
                        print(f"Predicted: {pred_str}")
                        print(f"Target   : {target_str}\n")

                total_samples += batch_size

        avg_loss = running_loss / len(self.val_loader)
        avg_cer = total_cer / total_samples if total_samples > 0 else 0.0
        return avg_loss, avg_cer


In [42]:
# Paths to test data
test_ground_truth_path = os.path.join(base_dir, '/kaggle/input/balinese/data/balinese_transliteration_test.txt')
test_images_dir = os.path.join(base_dir, '/kaggle/input/balinese/data/balinese_word_test')

test_filenames = []
test_labels = []

# Read the ground truth text file for the test set
with open(test_ground_truth_path, 'r', encoding='utf-8') as file:
    for line in file:
        line = line.strip()
        if line:
            parts = line.split(';')
            if len(parts) == 2:
                filename, label = parts
                test_filenames.append(filename)
                test_labels.append(label)
            else:
                print(f"Skipping malformed line: {line}")

# Create a DataFrame from the test data
test_data = pd.DataFrame({
    'filename': test_filenames,
    'label': test_labels
})


In [43]:
test_chars = set(''.join(test_data['label']))
unknown_chars = test_chars - set(char_to_idx.keys())
print(f"Unknown characters in test labels: {unknown_chars}")

Unknown characters in test labels: {'H'}


In [44]:
empty_test_labels = test_data[test_data['label'].str.strip() == '']
print(f"Number of empty labels in test data: {len(empty_test_labels)}")

Number of empty labels in test data: 0


In [45]:
max_label_length_test = max(len(label) for label in test_data['label']) + 2


def encode_label_test(label, char_to_idx, max_length):
    encoded = [char_to_idx['<SOS>']] + [char_to_idx.get(char, char_to_idx['<UNK>']) for char in label] + [char_to_idx['<EOS>']]
    if len(encoded) > max_length:
        encoded = encoded[:max_length]
    else:
        encoded += [char_to_idx['<PAD>']] * (max_length - len(encoded))
    return encoded


# Apply the encoding to the test data
test_data['encoded_label'] = test_data['label'].apply(lambda x: encode_label(x, char_to_idx, max_label_length_test))


In [46]:
# Define the same transformations used during training
test_transform = transforms.Compose([
    transforms.Resize((64, 256)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Create test dataset
test_dataset = BalineseDataset(test_data, test_images_dir, transform=test_transform)

# Create test data loader
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)


In [47]:
def inference(encoder, decoder, dataloader, device, char_to_idx, idx_to_char, max_seq_length):
    encoder.eval()
    decoder.eval()
    results = []

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(dataloader):
            images = images.to(device)
            batch_size = images.size(0)

            # Forward pass through encoder
            encoder_out = encoder(images)
            encoder_dim = encoder_out.size(-1)
            num_pixels = encoder_out.size(1)

            # Initialize LSTM state
            h, c = decoder.init_hidden_state(encoder_out)

            # Start with <SOS> token
            inputs = torch.full((batch_size,), char_to_idx['<SOS>'], dtype=torch.long).to(device)

            # Store predicted sequences
            predicted_seqs = []

            for _ in range(max_seq_length):
                embeddings = decoder.embedding(inputs)

                attention_weighted_encoding, alpha = decoder.attention(encoder_out, h)

                gate = decoder.sigmoid(decoder.f_beta(h))
                attention_weighted_encoding = gate * attention_weighted_encoding

                h, c = decoder.decode_step(
                    torch.cat([embeddings, attention_weighted_encoding], dim=1),
                    (h, c)
                )

                preds = decoder.fc(decoder.dropout(h))
                _, preds_idx = preds.max(1)

                predicted_seqs.append(preds_idx.cpu().numpy())

                inputs = preds_idx

            # Convert predictions to text
            predicted_seqs = np.array(predicted_seqs).T

            for i in range(batch_size):
                pred_indices = predicted_seqs[i]

                # Stop at <EOS>
                if char_to_idx['<EOS>'] in pred_indices:
                    eos_index = np.where(pred_indices == char_to_idx['<EOS>'])[0][0]
                    pred_indices = pred_indices[:eos_index]
                else:
                    pred_indices = pred_indices

                # Decode indices to characters
                pred_chars = [idx_to_char.get(idx, '') for idx in pred_indices]

                # Join characters to form strings
                pred_str = ''.join(pred_chars)

                # Get ground truth label (remove <SOS> and <EOS>)
                label_indices = labels[i].cpu().numpy()
                label_indices = label_indices[1:]  # Remove <SOS>
                if char_to_idx['<EOS>'] in label_indices:
                    eos_index = np.where(label_indices == char_to_idx['<EOS>'])[0][0]
                    label_indices = label_indices[:eos_index]
                else:
                    label_indices = label_indices[label_indices != char_to_idx['<PAD>']]

                label_chars = [idx_to_char.get(idx, '') for idx in label_indices]
                label_str = ''.join(label_chars)

                # Collect results
                image_filename = test_data.iloc[batch_idx * batch_size + i]['filename']
                results.append({
                    'image_filename': image_filename,
                    'predicted_caption': pred_str,
                    'ground_truth_caption': label_str
                })

    return results



In [48]:
# Run inference on the test set
test_results = inference(encoder, decoder, test_loader, device, char_to_idx, idx_to_char, max_label_length_test)


In [49]:
def calculate_cer(reference, hypothesis):
    if len(reference) == 0:
        return float('inf') if len(hypothesis) > 0 else 0.0
    return editdistance.eval(reference, hypothesis) / len(reference)

# Compute CER for each sample
cer_scores = []
for result in test_results:
    cer = calculate_cer(result['ground_truth_caption'], result['predicted_caption'])
    cer_scores.append(cer)


# Initialize total edit distance and total reference length
total_edit_distance = 0
total_reference_length = 0

for i, result in enumerate(test_results):
    reference = result['ground_truth_caption']
    hypothesis = result['predicted_caption']
    
    # Calculate edit distance
    edit_distance = editdistance.eval(reference, hypothesis)
    total_edit_distance += edit_distance
    total_reference_length += len(reference)
    
    # Optionally, compute per-sample CER and store it
    if len(reference) > 0:
        cer = edit_distance / len(reference)
    else:
        cer = 0.0  # Avoid division by zero
    test_results[i]['cer'] = cer  # Add CER to each result dictionary if needed

# Compute total CER
if total_reference_length > 0:
    total_cer = total_edit_distance / total_reference_length
else:
    total_cer = 0.0  # Avoid division by zero

print(f"Total CER on Test Set: {total_cer:.4f}")

average_cer = sum(cer_scores) / len(cer_scores)
print(f"Average CER on Test Set: {average_cer:.4f}")



Total CER on Test Set: 0.2115
Average CER on Test Set: 0.2014


In [50]:
# Compute CER for each sample and add it to test_results
cer_scores = []
for i, result in enumerate(test_results):
    cer = calculate_cer(result['ground_truth_caption'], result['predicted_caption'])
    cer_scores.append(cer)
    test_results[i]['cer'] = cer  # Add CER to each result dictionary

# Compute average CER
average_cer = sum(cer_scores) / len(cer_scores)
print(f"Average CER on Test Set: {average_cer:.4f}")

# Sort test_results by CER in descending order to find samples with highest errors
sorted_results = sorted(test_results, key=lambda x: x['cer'], reverse=True)

# Print top N samples with highest CER
N = 30
print(f"\nTop {N} samples with highest CER:")
for i in range(N):
    result = sorted_results[i]
    print(f"Sample {i + 1}:")
    print(f"Image Filename: {result['image_filename']}")
    print(f"CER: {result['cer']:.4f}")
    print(f"Ground Truth Caption: {result['ground_truth_caption']}")
    print(f"Predicted Caption  : {result['predicted_caption']}\n")


Average CER on Test Set: 0.2014

Top 30 samples with highest CER:
Sample 1:
Image Filename: test4934.png
CER: 11.0000
Ground Truth Caption: .
Predicted Caption  : carik-agung

Sample 2:
Image Filename: test10070.png
CER: 9.0000
Ground Truth Caption: .
Predicted Caption  : liawasnia

Sample 3:
Image Filename: test1330.png
CER: 8.0000
Ground Truth Caption: .
Predicted Caption  : tangkala

Sample 4:
Image Filename: test580.png
CER: 6.0000
Ground Truth Caption: .
Predicted Caption  : tiaria

Sample 5:
Image Filename: test1335.png
CER: 6.0000
Ground Truth Caption: .
Predicted Caption  : panapa

Sample 6:
Image Filename: test7927.png
CER: 6.0000
Ground Truth Caption: .
Predicted Caption  : tuatia

Sample 7:
Image Filename: test4766.png
CER: 5.0000
Ground Truth Caption: i
Predicted Caption  : baring

Sample 8:
Image Filename: test7454.png
CER: 5.0000
Ground Truth Caption: .
Predicted Caption  : riwan

Sample 9:
Image Filename: test8296.png
CER: 5.0000
Ground Truth Caption: .
Predicted Caption

In [51]:
# Additionally, print some random samples to get a general idea of model performance
import random

M = 5  # Number of random samples to display
random_samples = random.sample(test_results, M)
print(f"\nRandom {M} samples from Test Set:")
for i, result in enumerate(random_samples):
    print(f"Sample {i + 1}:")
    print(f"Image Filename: {result['image_filename']}")
    print(f"CER: {result['cer']:.4f}")
    print(f"Ground Truth Caption: {result['ground_truth_caption']}")
    print(f"Predicted Caption  : {result['predicted_caption']}\n")


Random 5 samples from Test Set:
Sample 1:
Image Filename: test8479.png
CER: 0.0000
Ground Truth Caption: rudita
Predicted Caption  : rudita

Sample 2:
Image Filename: test1976.png
CER: 0.0000
Ground Truth Caption: ,
Predicted Caption  : ,

Sample 3:
Image Filename: test3078.png
CER: 0.0000
Ground Truth Caption: .
Predicted Caption  : .

Sample 4:
Image Filename: test1970.png
CER: 0.0000
Ground Truth Caption: hana
Predicted Caption  : hana

Sample 5:
Image Filename: test4748.png
CER: 0.5000
Ground Truth Caption: geni
Predicted Caption  : reng

